In [ ]:
!git clone https://github.com/dave123981/commerce-recommendation-engine.git
%cd commerce-recommendation-engine
!pip install -r requirements.txt -q

fatal: destination path 'commerce-recommendation-engine' already exists and is not an empty directory.
/content/commerce-recommendation-engine


In [2]:
!pwd

!ls

/content/commerce-recommendation-engine
data  notebooks  README.md  requirements.txt  src


In [3]:
!pip install -r requirements.txt

In [ ]:
!pip install -q kagglehub
import kagglehub
kagglehub.login()   # small browser popup, only needed once per session

!python data/download_data.py          # kagglehub is now the default method
!python data/build_interactions.py

100% 197M/197M [00:01<00:00, 126MB/s]
Extracting files...
kagglehub cached the dataset at /root/.cache/kagglehub/datasets/psparks/instacart-market-basket-analysis/versions/1
Copied CSVs to /content/commerce-recommendation-engine/data/raw
Built 32,664,602 interactions | 175,072 users | 42,529 items | 3,229,920 orders
Saved to /content/commerce-recommendation-engine/data/interactions.csv
Kaggle credentials set.


In [5]:
import pandas as pd

interactions = pd.read_csv("data/interactions.csv", parse_dates=["timestamp"])

print(interactions.shape)
print(interactions.nunique())
print("sparsity:", 1 - len(interactions) / (interactions.user_id.nunique() * interactions.item_id.nunique()))
print("orders per user:", interactions.groupby("user_id").order_id.nunique().describe())
print("purchases per item:", interactions.groupby("item_id").order_id.nunique().describe())

(32664602, 5)
user_id       175072
item_id        42529
order_id     3229920
timestamp        366
quantity           1
dtype: int64
sparsity: 0.9956129204775982
orders per user: count    175072.000000
mean         18.449095
std          17.151432
min           2.000000
25%           7.000000
50%          12.000000
75%          23.000000
max         100.000000
Name: order_id, dtype: float64
purchases per item: count     42529.000000
mean        768.054786
std        5226.329338
min          10.000000
25%          29.000000
50%          87.000000
75%         336.000000
max      476386.000000
Name: order_id, dtype: float64


In [6]:
import data.build_interactions as bi
interactions = bi.build_interactions(min_orders_per_user=10, min_purchases_per_item=50)
interactions.to_csv("data/interactions.csv", index=False)

In [7]:
from src.metrics import time_based_split

train, test = time_based_split(interactions, test_frac=0.2)
train.to_csv("data/train.csv", index=False)
test.to_csv("data/test.csv", index=False)
print(len(train), len(test))

22297249 5520463


In [9]:
from src.v1_popularity import PopularityRecommender
from src.metrics import evaluate_model

model = PopularityRecommender(half_life_days=30).fit(train)
metrics = evaluate_model(model, test, k=10)
print(metrics)

{'precision@10': 0.042816720496663165, 'recall@10': 0.014891039007152738, 'map@10': 0.017301025888708167, 'ndcg@10': 0.04700983159132022}


In [13]:
import csv
from pathlib import Path

results_path = Path("results.csv")
write_header = not results_path.exists()
with open(results_path, "a", newline="") as f:
    writer = csv.writer(f)
    if write_header:
        writer.writerow(["version", "precision@10", "recall@10", "map@10", "ndcg@10"])
    writer.writerow(["v1_popularity", metrics["precision@10"], metrics["recall@10"], metrics["map@10"], metrics["ndcg@10"]])

In [12]:
#model check against random sampling
users_who_bought_top_item = set(train[train.item_id == top_item].user_id)
eligible_users = set(test.user_id) - users_who_bought_top_item

eligible_test = test[test.user_id.isin(eligible_users)]
hit_rate_eligible = eligible_test.groupby("user_id").item_id.apply(lambda x: top_item in set(x)).mean()
print(f"eligible users: {len(eligible_users)} | hit rate among them: {hit_rate_eligible:.4f}")

eligible users: 63065 | hit rate among them: 0.0526
